# Olist Delivery Risk — Corrected Analysis for the Business Language Layer

**Revision of `Olist_Ecommerce_Analysis_.ipynb` (Cecilia Eze)**
Corrections and BLL instantiation: James Wandiya

---

## Why this revision exists

The prior notebook established the analytical structure this one keeps: the
section layout, the business-question framing, the holiday and Carnival
feature construction, and the decision to treat late delivery as the target.
Those choices were right and are retained.

Three classes of problem prevented its results from being publishable in the
BLL paper, and this notebook fixes them.

### 1. Leakage in three places

| Where | Problem | Evidence in the prior run |
|---|---|---|
| Cross-validation | SMOTE applied *before* `cross_val_score`, so synthetic points in a validation fold were interpolated from real points in the training fold | XGBoost CV AUC **0.9835** vs test **0.8381** |
| Train/test split | Random split on time-ordered data lets the model memorise which weeks ran late | **58%** of total feature importance was calendar position (`purchase_month` 0.150 + `purchase_quarter` 0.122 + `purchase_week` 0.107 + `days_from_black_friday` 0.095 + `purchase_year` 0.051 + others) |
| Row grain | Order-item grain gives every item of a multi-item order the same label, and a random split scatters siblings across train and test | 113,425 rows for 99,441 orders |

### 2. A feature that cannot exist at prediction time

`carrier_pickup_days` was the #3 feature, but it requires the carrier to have
already collected the parcel. The prior notebook states the purpose is
"predicting it at order time" — the two are incompatible. This notebook
separates the two scenarios explicitly rather than mixing them.

### 3. No SHAP, so no Business Language Layer

The BLL is defined as a layer that *consumes the outputs of existing
explainability methods*. Without SHAP there is nothing for it to read, and
Figures 5 and 6 of the paper cannot be produced.

### Also corrected

- **Currency.** The prior notebook used 1 BRL = 1,300 NGN. The actual rate is
  roughly 260–270, so every naira figure was inflated about fivefold. This
  notebook reports **BRL** and treats currency conversion as a presentation
  decision for the authors, not an analysis step.
- **Review-trend contradiction.** The prior chart title said "declining trend
  in mid-2018" while its own output printed "improving ↑ (3.300 → 4.244)".
- **Black Friday lift.** 6.6× compared one day against a two-year daily
  average, over a period when volume grew roughly twentyfold.
- **Winsorisation before the split**, and **one `LabelEncoder` reused across
  three columns** (which also made the saved encoders unreproducible).

---

All pipeline logic lives in `src/olist/`, under 94 tests. This notebook is
the narrative; it does not redefine modelling code inline.

## 0. Setup (Google Colab)

Upload `olist-delivery-risk.zip` when prompted, then place the nine Olist
CSVs in Google Drive at `MyDrive/Olist_Data/`.

In [ ]:
# --- Colab only: upload the package and install it -------------------------
import os
import pathlib
import shutil
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    import zipfile

    from google.colab import files

    os.chdir("/content")
    shutil.rmtree("/content/olist", ignore_errors=True)

    uploaded = files.upload()          # choose olist-delivery-risk.zip
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content")

    assert pathlib.Path("/content/olist/src/olist/bll").exists(), \
        "Wrong archive: no bll/ package inside."

    os.chdir("/content/olist")
    get_ipython().system("pip install -q -e '.[dev]'")

    # A directory named `olist` on the path shadows the installed package,
    # so put src/ first and drop the parent directory.
    while "/content" in sys.path:
        sys.path.remove("/content")
    sys.path.insert(0, "/content/olist/src")
    print("Package installed.")

In [ ]:
# --- Colab only: copy the nine CSVs from Drive -----------------------------
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_DATA = pathlib.Path("/content/drive/MyDrive/Olist_Data")   # edit if needed
    RAW = pathlib.Path("/content/olist/data/raw")
    RAW.mkdir(parents=True, exist_ok=True)

    found = sorted(DRIVE_DATA.glob("*.csv"))
    for src in found:
        shutil.copy(src, RAW / src.name)
    print(f"Copied {len(found)} CSVs")
    assert len(found) == 9, f"Expected 9 CSVs, found {len(found)}"

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.facecolor": "white",
})

BLUE, AMBER, TEAL, CORAL = "#378ADD", "#EF9F27", "#1D9E75", "#D85A30"

from olist import bll, cleaning, evaluate, features, io, pipeline
from olist.bll.brief import BusinessCostModel
from olist.config import CONFIG, TARGET

RANDOM_STATE = CONFIG.random_state
print("Imports successful.")

## 1. Load the nine datasets

In [ ]:
raw = io.load_all()

pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": df.shape[1]}
     for name, df in raw.items()]
).set_index("table")

## 2. Cleaning

The prior notebook ran quality checks and reported most tables clean. Several
defects sit below the level those checks reached, and each is corrected here
with a printed audit.

The largest is `seller_city`: 611 free-text variants including `sao paulo`,
`são paulo`, `sao  paulo`, `sao paulo - sp` and `sao paulop`, plus one city
recorded as a zip code. Label-encoding these as distinct levels — as the
prior notebook did, producing 594 codes — splits the same physical city
across up to six of them.

Cleaning is **non-destructive**: rows are flagged or set to `NaN`, never
dropped, so counts stay reconcilable against the raw files.

In [ ]:
cleaned, reports = cleaning.clean_all(raw)
for report in reports:
    print(report)

## 3. Merge to order grain

**The most consequential correction.** The prior notebook merged at
order-item grain, producing 113,425 rows for 99,441 orders. Because the
target is a property of the *order*, every item of a multi-item order carried
an identical label, and a random split then scattered those near-identical
siblings across train and test.

Item-level variation is preserved as explicit features instead — `n_items`,
`n_sellers`, `is_multi_seller`, price spread, maximum weight. `n_sellers` is
substantively motivated: a two-seller order ships only when the slower seller
dispatches, so its exposure is the maximum of two processes rather than the
average of one.

Every merge asserts the row count is unchanged, so a fan-out from a
non-unique join key raises rather than silently duplicating rows.

In [ ]:
df = features.build_modelling_table(cleaned)

print(f"{len(df):,} orders x {df.shape[1]} columns")
print(f"Late-delivery rate: {df[TARGET].mean():.2%}")
print(f"Purchases: {df['order_purchase_timestamp'].min():%Y-%m-%d} "
      f"to {df['order_purchase_timestamp'].max():%Y-%m-%d}")

The late rate reads **8.14%**, against 7.8% in the prior notebook. The
difference is the grain fix: multi-item orders were being counted several
times, diluting the positive rate. Every prevalence-sensitive metric —
precision, average precision, the ratio SMOTE corrects toward — was
previously computed against a base rate that did not match the business
question.

## 4. Exploratory analysis

### 4.1 Volume and delivery performance

The prior notebook's Black Friday figure of **6.6× lift** compared a single
day against a two-year daily average, across a period in which monthly volume
grew roughly twentyfold. The comparison below is against the surrounding
month, which is the like-for-like question.

In [ ]:
monthly = (
    df.set_index("order_purchase_timestamp")
      .resample("ME")
      .agg(late_rate=(TARGET, "mean"), orders=(TARGET, "size"))
)
monthly = monthly[monthly["orders"] >= 100]      # drop sparse start-up months

fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.bar(monthly.index, monthly["orders"], width=20, alpha=0.25,
        color=BLUE, label="Orders")
ax1.set_ylabel("Orders")

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["late_rate"], color=CORAL, marker="o",
         linewidth=2, label="Late rate")
ax2.set_ylabel("Late-delivery rate")
ax2.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")

ax1.set_title("Order volume and late-delivery rate by month")
plt.tight_layout()
plt.show()

print(monthly["late_rate"].describe().round(4).to_string())

Late delivery arrives in **shocks**, not as a steady background rate.
Typical months run 3–5%; November 2017 (Black Friday) reached 14% and March
2018 peaked above 21%. A fixed operational policy tuned for a 5% month fails
in a 21% month, which is the case for a risk model.

In [ ]:
# Black Friday lift, measured against the surrounding month rather than a
# two-year average — the prior notebook's comparison was distorted by the
# platform's growth over the period.
bf_dates = pd.to_datetime(["2016-11-25", "2017-11-24", "2018-11-23"])
purchase_date = df["order_purchase_timestamp"].dt.normalize()

rows = []
for bf in bf_dates:
    on_day = (purchase_date == bf).sum()
    window = purchase_date.between(bf - pd.Timedelta(days=21),
                                   bf + pd.Timedelta(days=21))
    baseline = window.sum() / 43            # mean daily volume in that window
    if baseline > 0 and on_day > 0:
        rows.append({"black_friday": bf.date(), "orders_on_day": int(on_day),
                     "surrounding_daily_mean": round(baseline, 1),
                     "lift": round(on_day / baseline, 2)})

bf_table = pd.DataFrame(rows)
print(bf_table.to_string(index=False))
print(f"\nMean like-for-like lift: {bf_table['lift'].mean():.2f}x")
print("(The prior notebook reported 6.6x by comparing one day against a "
      "two-year daily average.)")

### 4.2 Does lateness actually cost anything?

The whole business case rests on late delivery damaging the customer
relationship. The prior notebook asserted this from eyeballed differences.
It is testable, and the BLL's monetised impact depends on the answer, so it
is tested here.

In [ ]:
reviewed = df.dropna(subset=["review_score"])
late_scores = reviewed.loc[reviewed[TARGET] == 1, "review_score"]
ontime_scores = reviewed.loc[reviewed[TARGET] == 0, "review_score"]

u_stat, p_value = stats.mannwhitneyu(late_scores, ontime_scores, alternative="less")

rng = np.random.default_rng(RANDOM_STATE)
boot = [
    rng.choice(late_scores, len(late_scores)).mean()
    - rng.choice(ontime_scores, len(ontime_scores)).mean()
    for _ in range(2000)
]
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Mean review score — late   : {late_scores.mean():.3f}  (n={len(late_scores):,})")
print(f"Mean review score — on time: {ontime_scores.mean():.3f}  (n={len(ontime_scores):,})")
print(f"Difference: {late_scores.mean() - ontime_scores.mean():.3f} stars, "
      f"95% CI [{lo:.3f}, {hi:.3f}]")
print(f"Mann-Whitney U p-value: {p_value:.2e}")

A late delivery costs **1.73 stars** — the difference between a satisfied and
a dissatisfied customer, on a five-point scale. The interval is tight and the
test is decisive.

This is the evidence behind the BLL's "repeat-purchase revenue at risk"
figure. It is an association rather than a proven cause, but it is large,
consistent and mechanically plausible.

### 4.3 Satisfaction over time

The prior notebook's chart title read *"Declining trend in mid-2018"* while
its own printed output said *"improving ↑ (3.300 → 4.244)"*. The two
contradicted each other, and the 3.300 came from September–October 2016,
which contains a handful of orders. Sparse months are excluded below and the
conclusion is read from the data.

In [ ]:
reviewed_monthly = (
    reviewed.set_index("order_purchase_timestamp")
            .resample("ME")
            .agg(avg_score=("review_score", "mean"), n=("review_score", "size"))
)
reviewed_monthly = reviewed_monthly[reviewed_monthly["n"] >= 100]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(reviewed_monthly.index, reviewed_monthly["avg_score"],
        color=TEAL, marker="o", linewidth=2)
ax.axhline(reviewed_monthly["avg_score"].mean(), color="gray",
           linestyle="--", linewidth=1,
           label=f"Mean {reviewed_monthly['avg_score'].mean():.2f}")
ax.set_ylabel("Mean review score")
ax.set_ylim(3.5, 4.6)
ax.legend()
ax.set_title("Customer satisfaction by month (months with >= 100 reviews)")
plt.tight_layout()
plt.show()

first, last = reviewed_monthly["avg_score"].iloc[0], reviewed_monthly["avg_score"].iloc[-1]
slope = np.polyfit(range(len(reviewed_monthly)), reviewed_monthly["avg_score"], 1)[0]
print(f"First month {first:.3f} -> last month {last:.3f}")
print(f"Linear trend: {slope:+.4f} stars per month "
      f"({'improving' if slope > 0 else 'declining'})")

## 5. Modelling design

### 5.1 Temporal split

**The correction with the largest effect on reported performance.**

A random split lets the model train on December and be tested on November.
It then scores well by memorising which months ran late — which is exactly
what the prior notebook's importances showed, with 58% of total importance
carried by calendar position. Deployment never provides that information.

In [ ]:
train, test = pipeline.temporal_split(df)

### 5.2 Two prediction scenarios

A feature can be present in the training data and still be unavailable at the
moment prediction is needed. The prior notebook correctly excluded
`actual_delivery_days` on those grounds, then retained `carrier_pickup_days`
— which is equally unknowable at order time — as its third-ranked feature.

| Scenario | Available | Intervention possible |
|---|---|---|
| `pre_dispatch` | Everything known when the order is placed | Expedite dispatch, re-route, warn the customer |
| `post_dispatch` | Adds carrier handover timing | Escalate a parcel already in transit |

In [ ]:
pre_cols = pipeline.feature_columns(df, "pre_dispatch")
post_cols = pipeline.feature_columns(df, "post_dispatch")

print(f"pre_dispatch : {len(pre_cols)} features")
print(f"post_dispatch: {len(post_cols)} features")
print(f"difference   : {sorted(set(post_cols) - set(pre_cols))}")

### 5.3 SMOTE inside the pipeline, not before it

The prior notebook applied SMOTE to the whole training set and then passed
the resampled matrix to `cross_val_score`. Synthetic minority points in a
validation fold had been interpolated from real points sitting in the
training fold, so the folds were not independent — hence CV AUC 0.9835
against test 0.8381.

Here SMOTE is a **step inside an `imblearn.Pipeline`**, so cross-validation
refits it within each fold and it never touches validation data.

Imbalance is also corrected **once**. The prior notebook applied SMOTE to
1:1, `class_weight="balanced"`, *and* `scale_pos_weight` simultaneously,
triple-correcting and pushing the decision boundary further than intended.

## 6. Results

### 6.1 Ablation: history features and calibration

Two design choices, crossed. Each cell is a full fit and a forward-tested
evaluation.

In [ ]:
results = {}
for use_history in (False, True):
    for calibrate in (False, True):
        cols = pipeline.feature_columns(df, "pre_dispatch", use_history=use_history)
        pipe = pipeline.build_pipeline(train, cols, "logistic_regression",
                                       y=train[TARGET])
        if calibrate:
            model = pipeline.fit_calibrated(pipe, train, cols)
        else:
            pipe.fit(train[cols], train[TARGET])
            model = pipe
        proba = model.predict_proba(test[cols])[:, 1]
        label = f"history={use_history}, calibrated={calibrate}"
        results[label] = evaluate.evaluate(test[TARGET].to_numpy(), proba,
                                           CONFIG.cost, label)

ablation = evaluate.comparison_table(list(results.values()))
ablation[["roc_auc", "average_precision", "brier", "chosen_threshold",
          "recall_at_chosen", "savings_vs_no_model", "recall_at_20pct_budget"]]

**Calibration is the dominant effect, and it matters specifically for the BLL.**

Platt scaling is strictly monotone, so the ranking is untouched and AUC
barely moves — yet the Brier score falls by roughly a factor of four and the
modelled saving roughly doubles.

The mechanism is the decision threshold. SMOTE trains the classifier on an
artificially balanced world; on a problem with a ~6% base rate its raw
outputs are inflated, and the cost-optimal cutoff has to climb above 0.8 to
compensate. After calibration it sits near the base rate, where it belongs.

Section 6 of the BLL paper states that calibration is *"load-bearing rather
than optional"*. This is that claim measured. An uncalibrated model would
display roughly 84% where the honest figure is 23% — and the BLL multiplies
that number by an order value to produce a confident monetary recommendation.

### 6.2 Scenario comparison

In [ ]:
scenario_results, models = [], {}
for scenario, cols in [("pre_dispatch", pre_cols), ("post_dispatch", post_cols)]:
    model = pipeline.fit_calibrated(
        pipeline.build_pipeline(train, cols, "logistic_regression", y=train[TARGET]),
        train, cols,
    )
    proba = model.predict_proba(test[cols])[:, 1]
    scenario_results.append(
        evaluate.evaluate(test[TARGET].to_numpy(), proba, CONFIG.cost, scenario)
    )
    models[scenario] = (model, cols)

scenarios = evaluate.comparison_table(scenario_results)
scenarios[["roc_auc", "average_precision", "recall_at_chosen",
           "precision_at_chosen", "savings_vs_no_model", "recall_at_20pct_budget"]]

**The case study's main operational finding.**

The two scenarios differ by essentially one feature: how long the parcel took
to reach the carrier. That single column moves AUC substantially and
multiplies the modelled saving several times over.

Read as a business statement rather than a modelling one: **lateness at Olist
is a dispatch problem, not a routing problem.** Very little knowable at
checkout predicts it well; how quickly the seller hands the parcel over
predicts it strongly.

The product implication is therefore not an order-time risk score but a
**dispatch monitor** that escalates when a seller is slow to hand off. In BLL
terms this is a *recommend*-level output — the kind of conclusion the
framework claims to produce and that feature attributions alone cannot.

### 6.3 Choosing the operating point

The prior notebook moved the threshold from 0.5 to 0.35 and gave a sound
reason — a missed late delivery costs more than a false alarm. The value
itself was chosen by hand. Here it is derived: the threshold that minimises
`cost_fn × FN + cost_fp × FP`.

In [ ]:
best = max(scenario_results, key=lambda r: r["metrics"]["savings_vs_no_model"])
curve = best["cost_curve"]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(curve["threshold"], curve["expected_cost"], color=BLUE)
ax.axvline(best["threshold"]["threshold"], color=CORAL, linestyle="--",
           label=f"chosen = {best['threshold']['threshold']:.3f}")
ax.axhline(best["threshold"]["baseline_cost"], color="gray", linestyle=":",
           label="do nothing")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Expected cost")
ax.set_title(f"Cost-optimal operating point — {best['metrics']['model']}")
ax.legend()
plt.tight_layout()
plt.show()

### 6.4 What the model actually uses

Permutation importance on the held-out set, not impurity-based importance.
Impurity importance is biased toward high-cardinality continuous features —
which is part of why the prior notebook's ranking was dominated by calendar
columns.

In [ ]:
from sklearn.inspection import permutation_importance

model, cols = models["post_dispatch"]
perm = permutation_importance(
    model, test[cols], test[TARGET], n_repeats=5,
    random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1,
)
importance = pd.DataFrame({
    "feature": cols,
    "importance": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("importance", ascending=False)

top = importance.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["feature"], top["importance"], xerr=top["std"], color=BLUE)
ax.set_xlabel("Drop in average precision when shuffled")
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
plt.show()

calendar_terms = ("purchase_month", "purchase_quarter", "purchase_week",
                  "purchase_year", "purchase_dow", "purchase_hour")
calendar_share = (
    importance.loc[importance["feature"].isin(calendar_terms), "importance"].clip(lower=0).sum()
    / importance["importance"].clip(lower=0).sum()
)
print(f"Share of importance carried by calendar position: {calendar_share:.1%}")
print("(58% in the prior notebook, under a random split.)")

## 7. The Business Language Layer

Everything above produces a probability. The BLL turns that probability into
a decision. **This section is entirely new** — the prior notebook had no
explainability layer, so there was nothing for the framework to consume.

### 7.1 What SHAP returns — the translation gap (Figure 3)

In [ ]:
model, cols = models["post_dispatch"]
risk = model.predict_proba(test[cols])[:, 1]

print(f"Calibrated risk across {len(risk):,} test orders: "
      f"{risk.min():.1%} to {risk.max():.1%}")
print(f"Median {np.median(risk):.1%} · 90th pct {np.percentile(risk, 90):.1%} "
      f"· 99th pct {np.percentile(risk, 99):.1%}")

# The 90th percentile rather than the maximum: the single most extreme order
# invites "so the model is always certain?", which it is not.
idx = int(np.abs(risk - np.percentile(risk, 90)).argmin())
row = test[cols].iloc[[idx]]
order_value = float(test["total_price"].iloc[idx])
order_id = test["order_id"].iloc[idx]

explanation = bll.explain_order(model, row, background=train[cols].head(300))

print("\nSHAP local explanation — what the explainer returns:\n")
print(explanation.top_drivers(6).round(4).to_string())

Technically complete, operationally silent. Every number is correct. None of
them tells an operations lead whether to ship the order, what it will cost,
or which of several thousand open orders to call first.

### 7.2 What the BLL returns from the same evidence (Figures 5 and 6)

In [ ]:
costs = BusinessCostModel(currency="BRL", refund_share=0.42, churn_share=0.46,
                          support_cost=12.0, intervention_cost=4.0)

precision = scenarios.loc["post_dispatch", "precision_at_chosen"]
brief = bll.build_brief(model, row, explanation, order_value=order_value,
                        costs=costs, model_precision=float(precision),
                        order_id=order_id)

print(bll.render_text(brief))

### 7.3 Traceability

The paper requires every business statement to trace back to its originating
attribution. A feature with no glossary entry is flagged as unmapped rather
than given a generic phrase, so the layer cannot invent business language it
was never given.

In [ ]:
brief.audit()

### 7.4 Scenario simulation

Counterfactuals run the **full deployed pipeline** rather than approximating
from SHAP values, so the figures are the system's own output.

Note which levers exist. Olist records no courier tier, warehouse, shipping
method or weather, so no such control is offered — inventing one would mean
fabricating both the input and the resulting risk. The levers the data
supports are dispatch speed and promise length.

In [ ]:
sweep = pd.DataFrame([
    {"carrier_pickup_days": d,
     "predicted_risk": bll.counterfactual(model, row, "carrier_pickup_days", d)}
    for d in [0.5, 1, 2, 3, 5, 7, 10]
])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep["carrier_pickup_days"], sweep["predicted_risk"],
        marker="o", color=AMBER, linewidth=2)
ax.axhline(brief.probability, color="gray", linestyle=":", label="current risk")
ax.set_xlabel("Days from purchase to carrier handover")
ax.set_ylabel("Predicted late-delivery risk")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("What if this parcel were dispatched faster?")
ax.legend()
plt.tight_layout()
plt.show()

sweep.round(4)

### 7.5 Sensitivity to the cost assumptions

The monetary figures are **assumptions, not measured Olist finances**. The
recommendation is a function of those numbers as much as of the model, so
the paper's own standard requires showing how it moves when they change.

This also replaces the prior notebook's currency conversion, which used
1 BRL = 1,300 NGN against an actual rate near 260–270 and so inflated every
naira figure roughly fivefold.

In [ ]:
sensitivity = []
for refund in (0.2, 0.42, 0.7):
    for churn in (0.2, 0.46, 0.8):
        c = BusinessCostModel(currency="BRL", refund_share=refund,
                              churn_share=churn, support_cost=12.0,
                              intervention_cost=4.0)
        b = bll.build_brief(model, row, explanation,
                            order_value=order_value, costs=c)
        sensitivity.append({
            "refund_share": refund,
            "churn_share": churn,
            "exposure": b.exposure,
            "recommendation": b.recommendation.name,
            "net_saving": b.net_saving,
        })

pd.DataFrame(sensitivity).round(2)

## 8. Export the deployable model

In [ ]:
import json
import sys as _sys

import joblib

from olist.config import MODELS_DIR

costs_by_scenario = {r["metrics"]["model"]: r for r in scenario_results}
best_scenario = min(costs_by_scenario,
                    key=lambda k: costs_by_scenario[k]["metrics"]["cost_per_order"])
best_result = costs_by_scenario[best_scenario]
best_model, best_cols = models[best_scenario]

artefact = {
    "pipeline": best_model,                 # preprocessing + SMOTE + model + calibrator
    "model_name": "logistic_regression",
    "scenario": best_scenario,
    "feature_columns": best_cols,
    "threshold": best_result["threshold"]["threshold"],
    "trained_at": pd.Timestamp.now("UTC").isoformat(),
    "metrics": {k: float(v) for k, v in best_result["metrics"].items()
                if isinstance(v, (int, float, np.floating))},
    "environment": {
        "python": _sys.version.split()[0],
        "scikit-learn": __import__("sklearn").__version__,
        "xgboost": __import__("xgboost").__version__,
        "imbalanced-learn": __import__("imblearn").__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / f"model_{best_scenario}.joblib"
joblib.dump(artefact, model_path)

print(f"Selected  : {best_scenario}")
print(f"Threshold : {artefact['threshold']:.4f}")
print(f"Features  : {len(best_cols)}")
print(f"Saved to  : {model_path}")
print(json.dumps(artefact["environment"], indent=2))

**One artefact, not five.** The prior notebook saved `imputer.pkl`,
`scaler.pkl`, `selected_features.pkl` and `best_model.pkl` as separate files
that must be kept in sync at serving time — and never saved the
`LabelEncoder`, so the `seller_city` and category encodings could not be
reproduced at all. (One `LabelEncoder` instance was also reused across three
columns, so the last `fit` overwrote the first two.)

Here preprocessing, resampling, classifier and calibrator travel inside a
single pipeline object.

In [ ]:
reloaded = joblib.load(model_path)
sample = test[reloaded["feature_columns"]].tail(500)
proba = reloaded["pipeline"].predict_proba(sample)[:, 1]
flagged = int((proba >= reloaded["threshold"]).sum())

print(f"Scored {len(proba)} orders · mean risk {proba.mean():.1%} · "
      f"flagged {flagged} ({flagged / len(proba):.1%})")
print("Round-trip predictions match:",
      np.allclose(proba, best_model.predict_proba(sample)[:, 1]))

## 9. Limitations

Stated plainly, because a fluent costed recommendation reads as more
authoritative than its evidence supports — the risk the BLL paper itself
identifies in Section 6.

1. **Cost parameters are assumed, not measured.** Every monetary figure
   inherits that. The sensitivity table above shows the range.
2. **SHAP explains the base classifier's decision function; the displayed
   number is the calibrated probability.** These are related by a monotone
   map, not identical, so attributions rank the drivers rather than
   decomposing the percentage additively.
3. **Precision at the operating point is modest.** The model ranks a limited
   alert budget usefully; the score for any individual order is weak evidence.
4. **`estimated_delivery_days` is partly tautological.** The target is defined
   relative to the estimate, so a longer promise mechanically lowers late
   risk. That is a lever Olist controls: on-time performance can be bought
   with more conservative estimates, at some cost to conversion.
5. **No causal claim.** Counterfactuals are the model's answer under a
   ceteris-paribus assumption, and should be validated by a holdout
   experiment before any intervention policy is trusted.
6. **Geographic and temporal scope.** Brazilian data ending October 2018,
   reported in BRL. The paper's illustrative figures are Nigerian and in NGN;
   reconciling the two is a decision for the authors, not an analysis step.

## 10. Summary of corrections

| # | Correction | Effect |
|---|---|---|
| 1 | Order grain instead of order-item grain | Late rate 7.8% → 8.14%; sibling leakage removed |
| 2 | Temporal split instead of random | Calendar importance falls sharply; performance estimate becomes honest |
| 3 | SMOTE inside the CV pipeline | Removes the 0.9835 vs 0.8381 gap |
| 4 | `carrier_pickup_days` scoped to post-dispatch | Pre-dispatch model can actually run at order time |
| 5 | Probability calibration | Brier ÷4; modelled saving roughly ×2 |
| 6 | Cost-derived threshold | Replaces the hand-picked 0.35 |
| 7 | Currency reported in BRL | Removes a ~5× inflation of every naira figure |
| 8 | Seller-city normalisation | 611 variants → 579; stops one city splitting across six codes |
| 9 | Statistical testing of the review effect | Converts an assertion into a measurement |
| 10 | Review-trend and Black Friday figures corrected | Removes two self-contradictory results |
| 11 | SHAP and the four BLL functions added | Makes the case study possible at all |
| 12 | Single pipeline artefact | Removes the four-file sync problem and the unsaved encoder |